In [6]:
import pandas as pd
from google.colab import files

# Upload CSV file in Colab
uploaded = files.upload()


Saving Revenue.csv to Revenue.csv


In [23]:
import re
# Extract the uploaded file name
csv_file = list(uploaded.keys())[0]

def load_financial_data(csv_file):
    try:
        # Load the CSV file into a DataFrame and normalize column names (strip spaces and convert to lowercase)
        df = pd.read_csv(csv_file)
        df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")  # Clean column names

        # Print out the column names to debug
        print("Available columns in the data:", df.columns)

        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

def extract_query_components(query):
    # Using regex to extract company, year, and financial metric from the query
    company_match = re.search(r"(Microsoft|Tesla|Apple)", query, re.IGNORECASE)
    year_match = re.search(r"(2024|2023|2022)", query)
    metric_match = re.search(r"(total revenue|revenue|net income|total assets|total liabilities|liabilities|operating cash flow|cash flow)", query, re.IGNORECASE)

    if company_match and year_match and metric_match:
        company = company_match.group(0).capitalize()
        year = int(year_match.group(0))
        metric = metric_match.group(0).lower().replace(" ", "_")  # Convert to column name format
        return company, year, metric
    else:
        return None, None, None

def find_matching_column(df, metric):
    # Find columns that contain the metric as a substring (case-insensitive)
    matching_columns = [col for col in df.columns if metric in col.lower()]

    if matching_columns:
        return matching_columns[0]  # Return the first match
    else:
        return None

def simple_chatbot(user_query, df):
    company, year, metric = extract_query_components(user_query)

    if company and year and metric:
        # Find the matching column for the metric
        matching_column = find_matching_column(df, metric)

        if matching_column:
            # Search the DataFrame for the matching company and year, and retrieve the requested metric
            result = df[(df['company'] == company) & (df['year'] == year)]
            if not result.empty:
                return result.iloc[0][matching_column]
            else:
                return "No data available for the requested company and year."
        else:
            return f"Sorry, I couldn't find a matching column for '{metric}'."
    else:
        return "Sorry, I couldn't understand the query. Please specify the company, year, and financial metric correctly."

# Load financial data from uploaded file
financial_data = load_financial_data(csv_file)

# Check if the data was loaded successfully
if financial_data is not None:
    print("Welcome to the Financial Chatbot! Ask a financial question or type 'exit' to quit.")
else:
    print("Failed to load data. Exiting chatbot.")

while True:
    user_input = input("\nEnter your query or type 'exit' to quit : ").strip()

    if user_input.lower() == "exit":
        print("Goodbye! Have a great day! 👋")
        break

    response = simple_chatbot(user_input, financial_data)
    print(response)


Available columns in the data: Index(['company', 'year', 'total_revenue', 'net_income', 'total_assets',
       'total_liabilities', 'operating_cash_flow'],
      dtype='object')
Welcome to the Financial Chatbot! Ask a financial question or type 'exit' to quit.

Enter your query or type 'exit' to quit : what is the total revenue for tesla in 2022
$81,462

Enter your query or type 'exit' to quit : exit
Goodbye! Have a great day! 👋
